# Neologism internals (one word up close)
Train one new tied row `E[new]` on sentences where a phrase was replaced by `{w}`, then look inside:

1. **Training**: loss, p(NEW) at held-out phrase slots vs random control slots.
2. **The row itself**: where `E[new]` moves (cos to init / gold / phrase mean, nearest rows, 2-D path).
3. **The slot**: per layer, does the state AT the new token match the state at the end of the real phrase? (matched vs shuffled)
4. **Downstream**: are the next tokens processed / predicted as if the real phrase were there? (KL + cos by offset)
5. **Logit lens at the slot**: per layer, what the model thinks comes next, NEW vs real phrase.
6. **Earlier words**: does anything before the slot change? (Phase A: provably not. Phase B: measured drift.)
7. **Attention**: how much later tokens read the word, before vs after.

Every plot compares **before** (row = init) with **after** (trained), and the new word with the **real phrase**.
Uses the repo's Model / train / make_inits; data from data/words/<WORD>/. For many words at once see neologism_sweep.

In [ ]:
# ---- 0. paths + config ----
import os, sys, copy, random
from pathlib import Path
ROOT = next(q for p in [Path.cwd(), *Path.cwd().parents] for q in (p, p / "src/model_new_words") if (q / "model.py").exists() and (q / "experiments").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt

WORD      = "comp_very_big"  # any dir in data/words (see neologism_sweep for the list)
INIT      = "mean_h"         # mean_emb | mean_h | in_span | gold  (make_inits rescales mean_h / in_span to the mean row norm;
                             # unscaled mean_h is ~35x too long and NEW would win every position)
N_TRAIN   = 200
LR_NEW    = 5e-3
LR_REST   = 0.0              # 0 = Phase A (only E[new] learns); > 0 = Phase B (whole model)
STEPS     = 300
BS        = 16
LOG_EVERY = 20
SEED      = 0
N_INT     = 64               # eval sentences used for the internals
MODEL     = "roneneldan/TinyStories-33M"
DEVICE    = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(SEED); random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

In [ ]:
# ---- 1. repo code, model, data ----
from model import Model
from train import TrainConfig, train
from dataset import load_word, word_meta
from experiments.neologism import make_inits

m = Model.load(MODEL, device=DEVICE)
tok = m.tok
meta = word_meta(WORD)
train_rows = load_word(WORD, "train_pool", n=N_TRAIN)
eval_rows = load_word(WORD, "eval")
ctrl_rows = load_word(WORD, "control")[:N_INT]
new_id = m.add_row()
PHRASE = " " + meta["phrases"][0]
GOLD = meta.get("gold")
gold_id = None
if GOLD:
    g = tok.encode(GOLD, add_special_tokens=False)
    gold_id = g[0] if len(g) == 1 else None
cfg = TrainConfig(steps=STEPS, bs=BS, lr_new=LR_NEW, lr_rest=LR_REST, log_every=LOG_EVERY, seed=SEED)
print(f"{WORD}: phrases {meta['phrases']}  gold {GOLD!r}  train {len(train_rows)}  eval {len(eval_rows)}  "
      f"control {len(ctrl_rows)}  E {tuple(m.E.shape)}  new_id {new_id}  device {DEVICE}")

In [ ]:
# ---- 2. helpers ----
def make_pair(row):
    """Same text twice via m.encode_pair: identical left, NEW vs the phrase tokens, identical right.
    slot = index of NEW; end = index of the phrase's LAST token (aligned with slot)."""
    ids_new, ids_orig, slot, nr = m.encode_pair(row, new_id)
    return dict(new=ids_new, orig=ids_orig, slot=slot, end=len(ids_orig) - nr - 1, n_after=nr)

def name(i):
    return "<NEW>" if i == new_id else repr(tok.decode([i]))

@torch.no_grad()
def run(hf, seqs, attn=False):
    """Right-padded batch forward. Returns logits [B,T,V], hidden list (emb + each layer) [B,T,d], attentions."""
    ids, att = m.pad(seqs)
    out = hf(input_ids=ids, attention_mask=att, output_hidden_states=True, output_attentions=attn)
    return out.logits.float(), [h.float() for h in out.hidden_states], (out.attentions if attn else None)

def cos(a, b):
    return F.cosine_similarity(a, b, dim=-1)

def phrase_ids():
    return tok.encode(PHRASE, add_special_tokens=False)

rng = random.Random(SEED)
ev = [make_pair(r) for r in rng.sample(eval_rows, min(N_INT, len(eval_rows)))]
ev = [p for p in ev if p["n_after"] >= 1]
ctrl = [tok.encode(r["orig"], add_special_tokens=False) for r in ctrl_rows]
def show(ids): return "".join(" <NEW>" if i == new_id else tok.decode([i]) for i in ids).replace("\n", " ")
print(len(ev), "eval pairs; example:\n  new :", show(ev[0]["new"]), "\n  orig:", show(ev[0]["orig"]))

In [ ]:
# ---- 3. init + "before" snapshot ----
act = m.active_ids(train_rows + eval_rows + ctrl_rows)
inits, info = make_inits(m, train_rows, new_id, act, gold_id=gold_id)
init = inits.get(INIT, inits["mean_h"]).to(m.E)
m.set_row(new_id, init)
hf_before = copy.deepcopy(m.hf).eval()          # 33M params: cheap. Frozen copy = "before".
E_before = m.E.detach().clone()
print("init:", INIT, f" |init| = {init.norm().item():.2f}", "  mean |active row| =", round(info["row_norm"].item(), 3))

In [ ]:
# ---- 4. train, logging p(NEW) at the slot on eval + control ----
@torch.no_grad()
def logger(step, m, e, last_loss):
    lg, _, _ = run(m.hf, [p["new"] for p in ev])
    p_slot = torch.stack([lg[i, p["slot"] - 1].softmax(-1)[new_id] for i, p in enumerate(ev)])
    d = {"p_new_eval": p_slot.mean().item(), "p_new_eval_med": p_slot.median().item()}
    if ctrl:   # how often the model would say NEW where the real text has no such word
        lc, _, _ = run(m.hf, ctrl)
        d["p_new_ctrl"] = torch.cat([lc[i, :len(s) - 1].softmax(-1)[:, new_id] for i, s in enumerate(ctrl)]).mean().item()
    nb, _ = m.neighbors(e, k=5, active=act, mu=info["mu"], upto=new_id)
    d["nn"] = [name(int(i)) for i in nb]
    return d

traj, logs = train(m, train_rows, new_id, init, cfg, logger=logger)
E_after = m.E.detach().clone()
steps = [l["step"] for l in logs]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(steps[1:], [l["loss"] for l in logs[1:]], "o-"); ax[0].set(title="train loss", xlabel="step")
ax[1].plot(steps, [l["p_new_eval"] for l in logs], "o-", label="eval slots (mean)")
ax[1].plot(steps, [l["p_new_eval_med"] for l in logs], "--", label="eval slots (median)")
if ctrl: ax[1].plot(steps, [l["p_new_ctrl"] for l in logs], "s-", label="control, every position")
ax[1].set(title="p(NEW)", xlabel="step", yscale="log"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ---- 5. the row: where does E[new] go? ----
actc = act.cpu()
En = F.normalize(E_after[actc].float().cpu(), dim=-1)          # neighbours among rows the corpus uses
T = traj.float()
refs = {"init": T[0], "phrase mean": E_before[phrase_ids()].float().cpu().mean(0)}
if gold_id is not None: refs["gold " + repr(GOLD)] = E_before[gold_id].float().cpu()

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for k, v in refs.items(): ax[0].plot(steps, cos(T, v[None]).numpy(), "o-", label=k)
ax[0].set(title="cos(E[new]_t, ref)", xlabel="step"); ax[0].legend()
ax[1].plot(steps, T.norm(dim=1).numpy(), "o-")
ax[1].axhline(E_before[:new_id].norm(dim=1).median().item(), ls="--", c="gray", label="median |old row|")
ax[1].set(title="|E[new]_t|", xlabel="step"); ax[1].legend()

# 2-D: PCA of (trajectory + its final top-15 neighbours + refs), so the path is drawn among real words
nb = actc[(En @ F.normalize(T[-1], dim=0)).topk(15).indices]
pts = torch.cat([T, E_after[nb].float().cpu(), torch.stack(list(refs.values()))])
mu = pts.mean(0); U, S, V = torch.pca_lowrank(pts - mu, q=2, center=False)
P = (pts - mu) @ V
nT = len(T)
ax[2].plot(P[:nT, 0], P[:nT, 1], "-o", ms=3, c="C3", label="E[new] path")
ax[2].scatter(P[nT:nT + 15, 0], P[nT:nT + 15, 1], c="C0", s=12)
for j, i in enumerate(nb.tolist()): ax[2].annotate(tok.decode([i]), P[nT + j], fontsize=8)
for j, k in enumerate(refs): ax[2].scatter(*P[nT + 15 + j], marker="*", s=120, label=k)
ax[2].set(title="PCA: path among final neighbours"); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

print("nearest rows over training:")
for l in logs: print(f"  step {l['step']:>5}  p_new={l['p_new_eval']:.3f}  ", " ".join(l["nn"]))

In [ ]:
# ---- 6. the slot: does h(NEW) look like h(end of real phrase)?  per layer, before vs after ----
# Raw cosines between GPT hidden states are all high (anisotropy), so each matched pair is compared
# with a SHUFFLED baseline: h(NEW in sentence i) vs h(phrase end in sentence j != i). States are centered
# by the per-layer mean of the real-phrase states, which removes the shared direction.
def slot_states(hf):
    _, Hn, _ = run(hf, [p["new"] for p in ev])
    _, Ho, _ = run(hf, [p["orig"] for p in ev])
    hn = torch.stack([torch.stack([H[i, p["slot"]] for i, p in enumerate(ev)]) for H in Hn])  # [L+1, N, d]
    ho = torch.stack([torch.stack([H[i, p["end"]] for i, p in enumerate(ev)]) for H in Ho])
    mu = ho.mean(1, keepdim=True)
    return (hn - mu).cpu(), (ho - mu).cpu()

S = {"before": slot_states(hf_before), "after": slot_states(m.hf)}
L1 = S["after"][0].shape[0]
perm = torch.roll(torch.arange(len(ev)), 1)

fig, ax = plt.subplots(1, 2, figsize=(14, 3.8), sharey=True)
for a, (k, (hn, ho)) in zip(ax, S.items()):
    matched = cos(hn, ho).numpy(); shuffled = cos(hn, ho[:, perm]).numpy()
    x = np.arange(L1)
    a.violinplot(list(matched), positions=x - .18, widths=.35, showmedians=True)
    a.violinplot(list(shuffled), positions=x + .18, widths=.35, showmedians=True)
    a.set(title=f"{k}: centered cos(h_NEW, h_phrase_end)   blue=matched, orange=shuffled", xlabel="layer (0 = embeddings)",
          xticks=x)
ax[0].set_ylabel("cosine")
plt.tight_layout(); plt.show()
for k, (hn, ho) in S.items():
    gap = (cos(hn, ho) - cos(hn, ho[:, perm])).mean(1)
    print(f"{k:>6}: matched - shuffled per layer:", np.round(gap.numpy(), 3))

In [ ]:
# ---- 7. downstream: are the following tokens processed as if the real phrase were there? ----
K = 5
def downstream(hf):
    """Per offset k: KL(p_phrase || p_NEW) of the next-token dist, and per-layer cos of the states.
    Only sentences with > k tokens after the word contribute to offset k."""
    ln, Hn, _ = run(hf, [p["new"] for p in ev])
    lo, Ho, _ = run(hf, [p["orig"] for p in ev])
    kl = [[] for _ in range(K + 1)]; cs = [[[] for _ in range(L1)] for _ in range(K + 1)]
    for i, p in enumerate(ev):
        for k in range(min(K, p["n_after"] - 1) + 1):   # offset 0 = the word itself (predicts 1st token after)
            a, b = p["slot"] + k, p["end"] + k
            kl[k].append(F.kl_div(ln[i, a].log_softmax(-1), lo[i, b].log_softmax(-1), log_target=True,
                                  reduction="sum").item())
            for l in range(L1): cs[k][l].append(cos(Hn[l][i, a], Ho[l][i, b]).item())
    return kl, np.array([[np.median(c) for c in row] for row in cs])

D = {"before": downstream(hf_before), "after": downstream(m.hf)}
fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
for j, (k, (kl, cs)) in enumerate(D.items()):
    ax[0].boxplot(kl, positions=np.arange(K + 1) + (j - .5) * .35, widths=.3,
                  patch_artist=True, boxprops=dict(facecolor=f"C{j}", alpha=.5))
    ax[0].plot([], [], c=f"C{j}", lw=6, alpha=.5, label=k)
    im = ax[1 + j].imshow(cs, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax[1 + j].set(title=f"{k}: median cos(h_new, h_orig)", xlabel="layer", ylabel="offset after word")
    plt.colorbar(im, ax=ax[1 + j])
ax[0].set(title="KL(p_phrase || p_NEW), next token", xlabel="offset after word", yscale="log",
          xticks=range(K + 1), xticklabels=range(K + 1)); ax[0].legend()
plt.tight_layout(); plt.show()
print("n sentences per offset:", [len(x) for x in D["after"][0]])
for k, (kl, _) in D.items(): print(f"{k:>6} median KL by offset:", np.round([np.median(x) for x in kl], 3))

In [ ]:
# ---- 8. logit lens at the slot: what does each layer predict next? ----
@torch.no_grad()
def lens(hf, seq, pos, top=3):
    W = hf.get_input_embeddings().weight.float()     # each model decodes with ITS OWN E (before != after for row new)
    _, H, _ = run(hf, [seq])
    rows = []
    for l, h in enumerate(H):
        x = h[0, pos] if l == len(H) - 1 else hf.transformer.ln_f(h[0, pos])   # GPT-Neo; last state already has ln_f
        rows.append(" ".join(f"{name(i)}:{p:.2f}" for p, i in zip(*(x @ W.T).softmax(-1).topk(top))))
    return rows

for p in ev[:3]:
    print("\n" + tok.decode(p["orig"]))
    A, B, C = lens(hf_before, p["new"], p["slot"]), lens(m.hf, p["new"], p["slot"]), lens(m.hf, p["orig"], p["end"])
    print(f"{'layer':<6}{'NEW before':<42}{'NEW after':<42}real phrase (after)")   # last layer = the model's actual output
    for l in range(len(A)): print(f"{l:<6}{A[l]:<42}{B[l]:<42}{C[l]}")

In [ ]:
# ---- 9. earlier words: did anything BEFORE the slot change? ----
# Phase A: weights frozen + causal attention => positions < slot are bit-identical. Checked, not assumed.
# Phase B: the body trained, so every word's representation can drift; measured on control sentences.
@torch.no_grad()
def prefix_drift(seqs, cut):
    _, Hb, _ = run(hf_before, seqs); _, Ha, _ = run(m.hf, seqs)
    out = []
    for l in range(L1):
        d = [((Ha[l][i, :c] - Hb[l][i, :c]).norm(dim=-1) / Hb[l][i, :c].norm(dim=-1).clamp_min(1e-6))
             for i, c in enumerate(cut) if c > 0]
        out.append(torch.cat(d).cpu().numpy())
    return out

pre = prefix_drift([p["new"] for p in ev], [p["slot"] for p in ev])
print("max relative change before the slot, per layer:", np.round([x.max() for x in pre], 6))
if ctrl:
    cd = prefix_drift(ctrl, [len(s) for s in ctrl])
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
    ax[0].violinplot(cd, positions=range(L1), showmedians=True)
    ax[0].set(title="control sentences: |h_after - h_before| / |h_before|", xlabel="layer")
    dE = (E_after[:new_id] - E_before[:new_id]).norm(dim=1).cpu()
    ax[1].hist(dE.numpy(), bins=60, log=True)
    ax[1].set(title="old rows: |E_after - E_before|", xlabel="drift")
    plt.tight_layout(); plt.show()
    top = dE.topk(10).indices.tolist()
    print("most-moved old rows:", ", ".join(f"{name(i)} {dE[i]:.2e}" for i in top))

In [ ]:
# ---- 10. attention: how much do later tokens read the slot? ----
def attn_to_slot(hf, which):
    try:
        hf.config._attn_implementation = "eager"          # sdpa/flash don't return weights
        _, _, A = run(hf, [p[which] for p in ev], attn=True)
    except Exception as e:
        print("attention weights unavailable:", e); return None
    out = torch.zeros(len(A), len(ev))
    for l, a in enumerate(A):                               # a: [B, heads, T, T]
        for i, p in enumerate(ev):
            s, e_ = (p["slot"], p["slot"]) if which == "new" else (p["slot"], p["end"])
            q0 = e_ + 1; q1 = q0 + p["n_after"]
            out[l, i] = a[i, :, q0:q1, s:e_ + 1].sum(-1).mean().item()   # mass on the word/phrase, avg heads+queries
    return out

R = {"NEW before": attn_to_slot(hf_before, "new"), "NEW after": attn_to_slot(m.hf, "new"),
     "phrase (after)": attn_to_slot(m.hf, "orig")}
if all(v is not None for v in R.values()):
    plt.figure(figsize=(7, 3.4))
    for j, (k, v) in enumerate(R.items()):
        plt.boxplot(list(v.numpy()), positions=np.arange(v.shape[0]) + (j - 1) * .27, widths=.24,
                    patch_artist=True, boxprops=dict(facecolor=f"C{j}", alpha=.5))
        plt.plot([], [], c=f"C{j}", lw=6, alpha=.5, label=k)
    plt.title("attention mass from later tokens onto the word"); plt.xlabel("layer"); plt.legend()
    plt.xticks(range(v.shape[0]), range(v.shape[0])); plt.tight_layout(); plt.show()